# Jørgensen, Teigen & Moløkken 2002 — Over-Confidence in Judgement Based Software Development Effort Prediction Intervals

In [1]:
import re, utils
from concurrent.futures import ThreadPoolExecutor, as_completed

LONG_DOC  = utils.load("full specification")
BRIEF_DOC = utils.load("project brief")

ROLES = {
    "EM":  "Engagement Manager — client relations; limited technical background",
    "PM":  "Project Manager — technical background; planning and delivery",
    "UD":  "User Interaction Designer — UX focus; not a core programmer",
    "DEV": "Software Developer — implementation",
}

def parse_pi(response):
    # Use extract_numbers to handle h/d suffixes and < separators
    def _get(label):
        m = re.search(rf"{label}:\s*([^\n]+)", response or "")
        if not m: return None
        nums = utils.extract_numbers(m.group(1))
        return int(nums[0]) if nums else None
    ml, mn, mx = _get("MOST_LIKELY"), _get("MINIMUM"), _get("MAXIMUM")
    return {"most_likely": ml, "minimum": mn, "maximum": mx,
            "pi_width": round((mx - mn) / ml, 3) if (ml and mn and mx and ml > 0) else None}


## Study A — 90% prediction interval hit rate

LLM provides a 90% PI per document. Hit rate is computed offline against TAWOS ground truth completion times.

In [ ]:
def build_study_a_tasks():
    tasks = []
    for doc_name, doc_text in LONG_DOC:
        prompt = f"""
You are a software developer estimating the effort required to complete the project below.
Assume average productivity for your team using the technology stack your company knows best.

Provide a 90% confidence prediction interval: you are 90% confident the actual effort will fall
between your minimum and maximum. That is, across 100 similar projects, the actual effort should
fall within your range approximately 90 times.

```requirements
{doc_text}
```

End your response with exactly:
MOST_LIKELY: <number> work-hours
MINIMUM: <number> work-hours
MAXIMUM: <number> work-hours
"""
        for model in utils.MODELS:
            tasks.append((model, prompt, {"doc": doc_name}))
    return tasks

def merge_study_a(store, model, response, meta):
    doc = meta["doc"]
    row = parse_pi(response)
    row["completion_times"] = utils.load_completion_times(doc)
    store.setdefault(model, {})[doc] = row
    if row["most_likely"] is None:
        utils.record_failure("jorgensen2002_a_failures.jsonl", model, response, meta)
        print(f"  FAIL {model.split('/')[1]:20} | {doc}")
    else:
        print(f"  OK   {model.split('/')[1]:20} | {doc} | ml={row['most_likely']}")

results_a = {}
utils.fire_and_collect(build_study_a_tasks(), results_a, merge_study_a,
                       save_path="jorgensen2002_a_partial.json")
utils.save("jorgensen2002_study_a_results.json", results_a)


## Study B — Role-based individual estimates and group consensus

Four roles estimate independently in parallel, then the group agrees on a consensus. Tests whether technical roles are more overconfident.

In [ ]:
# Study B requires two waves: all role estimates must finish before the group prompt
# can be built. We run wave 1 (roles), then wave 2 (group) via fire_and_collect.

ROLE_PROMPT = """You are a {desc}, estimating the total effort required to complete the project below.
Assume average productivity for your team. Provide a 90% prediction interval.
Note: although your role focuses on one area, you are estimating the total project effort.

```requirements
{doc_text}
```

End your response with exactly:
MOST_LIKELY: <number> work-hours
MINIMUM: <number> work-hours
MAXIMUM: <number> work-hours
"""

# ── Wave 1: all (doc × role × model) role estimates ──────────────────────────
wave1_tasks = [
    (model, ROLE_PROMPT.format(desc=desc, doc_text=doc_text),
     {"doc": doc_name, "role": role, "model": model})
    for doc_name, doc_text in LONG_DOC
    for role, desc in ROLES.items()
    for model in utils.MODELS
]

role_results = {}   # {doc_name: {model: {role: {ml,mn,mx,pi_width}}}}

def merge_wave1(store, model, response, meta):
    doc, role = meta["doc"], meta["role"]
    row = parse_pi(response)
    store.setdefault(doc, {}).setdefault(model, {})[role] = row
    if row["most_likely"] is None:
        utils.record_failure("jorgensen2002_b_failures.jsonl", model, response, meta)
        print(f"  FAIL W1 {model.split('/')[1]:20} | {doc:30} | {role}")
    else:
        print(f"  OK   W1 {model.split('/')[1]:20} | {doc:30} | {role} | ml={row['most_likely']}")

utils.fire_and_collect(wave1_tasks, role_results, merge_wave1)

# ── Wave 2: group consensus, one per (doc × model) ───────────────────────────
def build_wave2_tasks():
    tasks = []
    for doc_name, doc_text in LONG_DOC:
        for model in utils.MODELS:
            row   = role_results.get(doc_name, {}).get(model, {})
            reveal = "  ".join(
                f"{k}: ML={row.get(k,{}).get('most_likely','?')}h "
                f"PI=[{row.get(k,{}).get('minimum','?')}, {row.get(k,{}).get('maximum','?')}]"
                for k in ROLES)
            prompt = f"""
A four-person estimation team has independently estimated the project below and now discusses
to agree on a consensus. Individual estimates: {reveal}

```requirements
{doc_text}
```

End your response with exactly:
MOST_LIKELY: <number> work-hours
MINIMUM: <number> work-hours
MAXIMUM: <number> work-hours
"""
            tasks.append((model, prompt, {"doc": doc_name, "model": model}))
    return tasks

results_b = {}

def merge_wave2(store, model, response, meta):
    doc = meta["doc"]
    row = dict(role_results.get(doc, {}).get(model, {}))
    row["GROUP"] = parse_pi(response)
    store.setdefault(model, {})[doc] = row
    g = row["GROUP"]
    if g["most_likely"] is None:
        utils.record_failure("jorgensen2002_b_failures.jsonl", model, response, meta)
        print(f"  FAIL W2 {model.split('/')[1]:20} | {doc:30}")
    else:
        print(f"  OK   W2 {model.split('/')[1]:20} | {doc:30} → group={g['most_likely']}h  width={g['pi_width']}")

utils.fire_and_collect(build_wave2_tasks(), results_b, merge_wave2,
                       save_path="jorgensen2002_b_partial.json")
utils.save("jorgensen2002_study_b_results.json", results_b)


## Study C — Confidence level sensitivity

The same project brief is estimated at four confidence levels. PI width should increase markedly with confidence level if the LLM is well-calibrated.

In [ ]:
def build_study_c_tasks():
    tasks = []
    for doc_name, doc_text in BRIEF_DOC:
        for confidence in [50, 75, 90, 99]:
            prompt = f"""
You are a software developer estimating the effort required to complete the project below.
Assume average productivity for your team using the technology stack your company knows best.

Provide a {confidence}% prediction interval: you are {confidence}% confident the actual
effort will fall between your minimum and maximum.

```requirements
{doc_text}
```

End your response with exactly:
MOST_LIKELY: <number> work-hours
MINIMUM: <number> work-hours
MAXIMUM: <number> work-hours
"""
            for model in utils.MODELS:
                tasks.append((model, prompt, {"doc": doc_name, "confidence": confidence}))
    return tasks

def merge_study_c(store, model, response, meta):
    doc, conf = meta["doc"], meta["confidence"]
    row = parse_pi(response)
    store.setdefault(model, {}).setdefault(doc, {})[conf] = row
    if row["most_likely"] is None:
        utils.record_failure("jorgensen2002_c_failures.jsonl", model, response, meta)
        print(f"  FAIL {model.split('/')[1]:20} | {doc:30} | {conf}%")
    else:
        print(f"  OK   {model.split('/')[1]:20} | {doc:30} | {conf}% | ml={row['most_likely']}")

results_c = {}
utils.fire_and_collect(build_study_c_tasks(), results_c, merge_study_c,
                       save_path="jorgensen2002_c_partial.json")
utils.save("jorgensen2002_study_c_results.json", results_c)


## Study D — Ego-free prediction intervals (new experiment)

Motivation: developers give narrow PIs partly to signal competence. Framing estimation as a favour for a colleague removes that pressure. Compare PI width against Study A to test whether the bias has an ego-driven component.

In [ ]:
def build_study_d_tasks():
    tasks = []
    for doc_name, doc_text in LONG_DOC:
        prompt = f"""
A colleague has asked you to provide an independent effort estimate for their project.
You have no personal stake in the outcome — you are simply offering an outside perspective.
Provide a 90% prediction interval for how long the project is likely to take.

```requirements
{doc_text}
```

End your response with exactly:
MOST_LIKELY: <number> work-hours
MINIMUM: <number> work-hours
MAXIMUM: <number> work-hours
"""
        for model in utils.MODELS:
            tasks.append((model, prompt, {"doc": doc_name}))
    return tasks

def merge_study_d(store, model, response, meta):
    doc = meta["doc"]
    row = parse_pi(response)
    store.setdefault(model, {})[doc] = row
    if row["most_likely"] is None:
        utils.record_failure("jorgensen2002_d_failures.jsonl", model, response, meta)
        print(f"  FAIL {model.split('/')[1]:20} | {doc}")
    else:
        print(f"  OK   {model.split('/')[1]:20} | {doc} | ml={row['most_likely']}")

results_d = {}
utils.fire_and_collect(build_study_d_tasks(), results_d, merge_study_d,
                       save_path="jorgensen2002_d_partial.json")
utils.save("jorgensen2002_study_d_results.json", results_d)
